# Notebook 02: SQL to Python – Integrating EHR Data

In this notebook, you’ll work across SQL and Python. You’ll:

- Load synthetic EHR data from CSV files
- Create SQLite tables from your data
- Use SQL to query the tables
- Use pandas to perform similar operations
- Understand how SQL and pandas interact using `read_sql_query()`

In [ ]:
# Specify the disk drive path to the folder containing the Synthea CSV files
# You'll change the path on the next line to match your system setup
my_path = "/Users/spb24/Downloads/coherent-11-07-2022/csv/"

In [ ]:
import pandas
import sqlite3

# Load patient and encounter data from CSV
patients = pandas.read_csv(my_path + "patients.csv")
encounters = pandas.read_csv(my_path + "encounters.csv")

✅ **Checkpoint**  
Use the cells below to preview your dataframes with `.head()`.

In [ ]:
patients.head()

In [ ]:
encounters.head()

In [ ]:
# Create an in-memory SQLite database
rdbms = sqlite3.connect(":memory:")

# Write both dataframes into the database as SQL tables
patients.to_sql("patients", rdbms, index=False, if_exists="replace")
encounters.to_sql("encounters", rdbms, index=False, if_exists="replace")

### 🔹 SQL Query: Preview Patients Table

Use SQL to return the first 5 rows of the patients table.

In [ ]:
query = """
SELECT * FROM patients
LIMIT 5;
"""

pandas.read_sql_query(query, rdbms)

### 🔹 SQL Query: Join Patients and Encounters

This example shows how to combine information from two tables:
- `JOIN` links rows based on a shared key
- `patients.Id = encounters.PATIENT` matches each encounter to a patient
- `WHERE` filters to patients older than a threshold (age derived from birthdate)

In [ ]:
# First, calculate age and write it into the database
from datetime import datetime

patients['BIRTHDATE'] = pandas.to_datetime(patients['BIRTHDATE'], errors='coerce')
patients['age'] = (datetime.today() - patients['BIRTHDATE']).dt.days // 365
patients.to_sql("patients", rdbms, index=False, if_exists="replace")

In [ ]:
# SQL to find encounters for patients over age 65
query = """
SELECT e.PATIENT, e.ENCOUNTERCLASS, p.age
FROM encounters e
JOIN patients p ON e.PATIENT = p.Id
WHERE p.age > 65;
"""

pandas.read_sql_query(query, rdbms)

### 🔁 Equivalent in pandas

Here's how to achieve the same result using pandas:
- `.merge()` performs the join
- Boolean filtering selects patients over age 65
- You can select specific columns after filtering

In [ ]:
# Join using pandas and filter by age
result = (
    encounters.merge(patients, left_on='PATIENT', right_on='Id', how='inner')
    [lambda df: df['age'] > 65]
    [['PATIENT', 'ENCOUNTERCLASS', 'age']]
)

result.head()

✏️ **Reflect**  
What advantages or limitations do you notice when working with SQL versus pandas?